# Strukturierte Outputs mit Pydantic

Dieser Code zeigt ein Beispiel, wie wir LLMs dazu bringen können, Antworten in einer bestimmten JSON-Struktur zu formulieren. Damit können wir die LLM-Outputs mehr oder weniger direkt in einem anderen Programm weiterverarbeiten. Die in llm_beispiel_basics erklärten Schritte erkläre ich hier nicht nochmals, sie sind aber im Code enthalten.

In [2]:
import pandas as pd
import numpy as np
import ollama
import re
from pathlib import Path

#Zusätzliche Pakete
from pydantic import Field, BaseModel, ValidationError
from typing import List, Optional, Literal #muss evtl angepasst werden, je nach genutzten Typen
import json

# Texte einlesen
texts = [
    "Andy Burnham will bring new energy and ideas to the Labour Party. He has a proven track record of leadership and is committed to addressing the issues that matter most to the British people.",
    "Keir Starmer has been unable to solve the problems affecting Britain. He is lacking imagination and leadership, and it is time for him to make way for new faces.",
    "The government has been right in highlighting climate change as an important issue, but its Net Zero goal and the policies associated with it have been overly ambitious and caused significant economic harm."
]

## Pydantic-Modell definieren

Pydantic erlaubt uns, ein bestimmtes "Modell" (in diesem Zusammenhang: Antwortschema) zu definieren. Dieses Modell können wir dann als JSON-Schema speichern und in den Prompt einfügen. Wir definieren hier ein einfaches Modell mit drei Feldern: ob der Text negative campaigning enthält (eine Variable mit zwei möglichen Outputs), eine Angabe der Sicherheit als Zahl, und eine (fakultative) Begründung.

class definiert dabei eine Python-Klasse, ReponseScheme ist der Name und kann angepasst werden. (BaseModel) muss jeweils angehängt werden, um klar zu machen, dass wir ein Pydantic-Modell definieren wollen.

Für jedes gewünschte Feld definieren wir einen Namen, die Art des Inputs (hier: Literal, float, str), ob es fakultativ oder obligatorisch ist. Unter "Field()" können wir eine Beschreibung (description = ) angeben, sowie weitere Restriktionen angeben. Siehe die Pydantic-Dokumentation für mehr Informationen.

In [4]:
class ResponseScheme(BaseModel):
    negcamp: Literal['Yes', 'No'] = Field(..., description = "Whether there is negative campaigning present")
    certainty: float = Field(..., description = "Confidence in your decision")
    justification: Optional[str] = Field(None, description = "Why did you make your decision?")

resp_json = ResponseScheme.model_json_schema()

print(resp_json)

{'properties': {'negcamp': {'description': 'Whether there is negative campaigning present', 'enum': ['Yes', 'No'], 'title': 'Negcamp', 'type': 'string'}, 'certainty': {'description': 'Confidence in your decision', 'title': 'Certainty', 'type': 'number'}, 'justification': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Why did you make your decision?', 'title': 'Justification'}}, 'required': ['negcamp', 'certainty'], 'title': 'ResponseScheme', 'type': 'object'}


## Prompting

Wir müssen unseren Prompt so erweitern, dass wir dem LLM sagen, welchen Output wir erwarten.

In [5]:
sysprompt = f'''You are a political scientist with a deep knowledge of political campaigning, but no ideological preferences. You are tasked with determining whether a given statement made by a British political party contains an example of negative campaigning or not.

## Definition of Negative Campaigning

Negative campaigning refers to any instance where a political candidate or party attacks their opponent's character, record, or policies rather than promoting their own platform. Negative campaigning does not have to be uncivil or aimed at the character of the opponent, and also includes factual criticism of an opponent's policy.

## Instructions

You will be given a text statement made by a British political party. Determine whether the statement contains an example of negative campaigning or not. If it does, respond with "Yes". If it does not, respond with "No". Add a measure of how certain you are on a probability-like scale from 0 to 1. Add a short explanation for your decision.

### Output

Return a JSON using the following schema: {resp_json}. Return nothing else.
'''

## Setup und Inferenz

Wir geben dem Modell eine weitere Option mit: "format", die dem Modell sagt, was für einen Output wir erwarten.

In [8]:
modelname = "mistral:7b"

#ollama.pull(modelname)

client = ollama.Client()

out = [] #Leere Liste, um die Ergebnisse zu speichern

opts = {
    "seed": 42,
    "temperature": 0.7, #Wir benutzen hier eine eher hohe Temperatur; oft würde man auch 0.0 für Reproduzierbarkeit nehmen - siehe Präsentation
}

for text in texts:

    messages = [
        {"role": "system", "content": sysprompt},
        {"role": "user", "content": text}
    ]

    response = client.chat(
        model=modelname, 
        messages=messages, 
        options = opts,
        format = resp_json)

    out.append(response) #Füge die Antwort der Liste hinzu

#Was hat Mistral produziert? Rohe Ergebnisse ausgeben
print(out)

[ChatResponse(model='mistral:7b', created_at='2026-07-30T14:35:04.7892291Z', done=True, done_reason='stop', total_duration=31999048100, load_duration=11427603600, prompt_eval_count=413, prompt_eval_duration=7883319000, eval_count=52, eval_duration=12657180000, message=Message(role='assistant', content='{"negcamp": "No", "certainty": 1, "justification": "The statement focuses on promoting the candidate\'s qualities and policies, rather than attacking their opponents. It does not fit the definition of negative campaigning."}', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None), ChatResponse(model='mistral:7b', created_at='2026-07-30T14:35:20.0042871Z', done=True, done_reason='stop', total_duration=15212635200, load_duration=51289100, prompt_eval_count=409, prompt_eval_duration=828679000, eval_count=59, eval_duration=14325543000, message=Message(role='assistant', content='{"negcamp": "Yes", "certainty": 0.9, "justification": "The statement directly criticizes the

## Verifizierung und post-processing

Das LLM kann immer noch einen formell falschen Output geben. Pydantic erlaubt uns, das Modell zu validieren und nur diejenigen Antworten, die korrekt ausgegeben wurden, zu speichern. Wir lassen uns ausserdem Fehlermeldungen loggen. 

In [21]:
validated_outs = []

for answer in out:
    try:
        content = answer.message.content
        parsed = json.loads(content)
        validated = ResponseScheme(**parsed)
        validated_outs.append(validated.model_dump())
    except (ValidationError, json.JSONDecodeError, AttributeError) as e:
        print(f"Invalid Response: {answer}. Error: {e}")

print(validated_outs)

#Zum Speichern: 
with open("example_outs.json", "w", encoding="utf-8") as f:
    json.dump(validated_outs, f, indent=2, ensure_ascii=False)

[{'negcamp': 'No', 'certainty': 1.0, 'justification': "The statement focuses on promoting the candidate's qualities and policies, rather than attacking their opponents. It does not fit the definition of negative campaigning."}, {'negcamp': 'Yes', 'certainty': 0.9, 'justification': "The statement directly criticizes the opponent's (Keir Starmer) abilities as a leader and suggests he is unable to solve problems, which falls under negative campaigning."}, {'negcamp': 'Yes', 'certainty': 0.95, 'justification': "The statement criticizes the government's policies (Net Zero goal and associated policies) as causing significant economic harm, which is a form of negative campaigning."}]
